In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from delta.tables import DeltaTable

# Get the insert_timestamp parameter
insert_ts = dbutils.widgets.get("insert_timestamp")

# Determine load type
load_type = "FULL" if not insert_ts or insert_ts.strip() == "" else "INCREMENTAL"
print(f"Load Type: {load_type}")
if load_type == "INCREMENTAL":
    print(f"Loading data where insert_timestamp > {insert_ts}")

Load Type: INCREMENTAL
Loading data where insert_timestamp > 2026-06-22T15:33:15.365+00:00


In [0]:
# Read from bronze table with conditional logic
if load_type == "FULL":
    # Full load - read all data
    bronze_df = spark.table("data_modelling_demo.bronze.scd1")
    print(f"Full load: Read {bronze_df.count()} records from bronze table")
else:
    # Incremental load - read only new data
    bronze_df = spark.table("data_modelling_demo.bronze.scd1") \
        .filter(F.col("insert_timestamp") > insert_ts)
    print(f"Incremental load: Read {bronze_df.count()} records from bronze table")

# Display sample data
display(bronze_df.limit(5))

Incremental load: Read 3 records from bronze table


customer_id,first_name,email,city,registration_date,last_updated_ts,_rescued_data,insert_timestamp
C001,john,john.smith@gmail.com,chicago,10-Jan-2024,2024-02-01 09:30:00,null,2026-06-23T07:14:12.076Z
C002,alice,alice123@gmail.com,boston,11-Jan-2024,2024-02-01 10:45:00,null,2026-06-23T07:14:12.076Z
C006,emma,emma.thomas@gmail.com,san francisco,01-Feb-2024,2024-02-01 11:15:00,null,2026-06-23T07:14:12.076Z


In [0]:
# Apply transformations
transformed_df = bronze_df \
    .withColumn("first_name", F.initcap(F.col("first_name"))) \
    .withColumn("email", F.lower(F.col("email"))) \
    .withColumn("registration_date", F.to_date(F.col("registration_date"), "dd-MMM-yyyy"))

print(f"Transformations applied successfully")
display(transformed_df.limit(5))

Transformations applied successfully


customer_id,first_name,email,city,registration_date,last_updated_ts,_rescued_data,insert_timestamp
C001,John,john.smith@gmail.com,chicago,2024-01-10,2024-02-01 09:30:00,null,2026-06-23T07:14:12.076Z
C002,Alice,alice123@gmail.com,boston,2024-01-11,2024-02-01 10:45:00,null,2026-06-23T07:14:12.076Z
C006,Emma,emma.thomas@gmail.com,san francisco,2024-02-01,2024-02-01 11:15:00,null,2026-06-23T07:14:12.076Z


In [0]:
# Apply data quality checks
from pyspark.sql import DataFrame

def apply_data_quality_checks(df: DataFrame) -> DataFrame:
    """
    Apply data quality checks and filter out invalid records
    """
    # Count before quality checks
    count_before = df.count()
    
    # Data quality rules:
    # 1. Remove records with null customer_id (primary key)
    # 2. Remove records with null or empty first_name
    # 3. Remove records with invalid email (must contain @)
    # 4. Remove records with null registration_date
    
    quality_df = df \
        .filter(F.col("customer_id").isNotNull()) \
        .filter((F.col("first_name").isNotNull()) & (F.trim(F.col("first_name")) != "")) \
        .filter((F.col("email").isNotNull()) & (F.col("email").contains("@"))) \
        .filter(F.col("registration_date").isNotNull())
    
    # Count after quality checks
    count_after = quality_df.count()
    records_rejected = count_before - count_after
    
    print(f"Data Quality Check Results:")
    print(f"  Records before: {count_before}")
    print(f"  Records after: {count_after}")
    print(f"  Records rejected: {records_rejected}")
    
    return quality_df

# Apply quality checks
quality_df = apply_data_quality_checks(transformed_df)
display(quality_df.limit(5))

Data Quality Check Results:
  Records before: 3
  Records after: 3
  Records rejected: 0


customer_id,first_name,email,city,registration_date,last_updated_ts,_rescued_data,insert_timestamp
C001,John,john.smith@gmail.com,chicago,2024-01-10,2024-02-01 09:30:00,null,2026-06-23T07:14:12.076Z
C002,Alice,alice123@gmail.com,boston,2024-01-11,2024-02-01 10:45:00,null,2026-06-23T07:14:12.076Z
C006,Emma,emma.thomas@gmail.com,san francisco,2024-02-01,2024-02-01 11:15:00,null,2026-06-23T07:14:12.076Z


In [0]:
# Define silver table name
silver_table = "data_modelling_demo.silver.scd1"

# Add processing timestamp
final_df = quality_df.withColumn("processed_timestamp", F.current_timestamp())

# Check if silver table exists
if spark.catalog.tableExists(silver_table):
    print(f"Silver table '{silver_table}' exists. Performing SCD Type 1 MERGE...")
    
    # Load existing silver table as Delta table
    silver_delta = DeltaTable.forName(spark, silver_table)
    
    # SCD Type 1: MERGE on customer_id (primary key)
    # When matched: UPDATE all columns with new values (overwrite existing)
    # When not matched: INSERT new records
    silver_delta.alias("target").merge(
        final_df.alias("source"),
        "target.customer_id = source.customer_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    
    print("✓ MERGE completed successfully (SCD Type 1)")
    print("  - Existing records: UPDATED with latest values")
    print("  - New records: INSERTED")
else:
    print(f"Silver table '{silver_table}' does not exist. Creating new table...")
    
    # Create new silver table
    final_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(silver_table)
    
    print(f"✓ Silver table '{silver_table}' created successfully")

# Show table statistics
print(f"\nSilver table record count:")
silver_count = spark.table(silver_table).count()
print(f"Total records in silver: {silver_count}")

print("\nSample from silver table:")
display(spark.table(silver_table).orderBy(F.col("processed_timestamp").desc()).limit(10))

Silver table 'data_modelling_demo.silver.scd1' exists. Performing SCD Type 1 MERGE...
✓ MERGE completed successfully (SCD Type 1)
  - Existing records: UPDATED with latest values
  - New records: INSERTED

Silver table record count:
Total records in silver: 6

Sample from silver table:


customer_id,first_name,email,city,registration_date,last_updated_ts,_rescued_data,insert_timestamp,processed_timestamp
C006,Emma,emma.thomas@gmail.com,san francisco,2024-02-01,2024-02-01 11:15:00,null,2026-06-23T07:14:12.076Z,2026-06-24T11:58:26.699Z
C002,Alice,alice123@gmail.com,boston,2024-01-11,2024-02-01 10:45:00,null,2026-06-23T07:14:12.076Z,2026-06-24T11:58:26.699Z
C001,John,john.smith@gmail.com,chicago,2024-01-10,2024-02-01 09:30:00,null,2026-06-23T07:14:12.076Z,2026-06-24T11:58:26.699Z
C004,Sarah,sarah.davis@gmail.com,seattle,2024-01-13,2024-01-13 15:45:00,null,2026-06-22T15:33:15.365Z,2026-06-23T07:11:14.446Z
C003,Michael,michael.brown@gmail.com,chicago,2024-01-12,2024-01-12 09:10:00,null,2026-06-22T15:33:15.365Z,2026-06-23T07:11:14.446Z
C005,David,david.wilson@gmail.com,dallas,2024-01-14,2024-01-14 16:00:00,null,2026-06-22T15:33:15.365Z,2026-06-23T07:11:14.446Z
